In [6]:
import torch

In [7]:
torch.manual_seed(0)
x = torch.randn(2, 4, requires_grad=True)
w = torch.randn(2, 4)

y = torch.softmax(x, dim=-1)

loss = (y * w).sum()

loss.backward()
x.grad

tensor([[-0.1616,  0.0065, -0.0020,  0.1571],
        [-0.0425,  0.0790, -0.2612,  0.2247]])

In [8]:
torch.manual_seed(0)
x = torch.randn(2, 4, requires_grad=True)
w = torch.randn(2, 4)

m = x.max(dim=-1, keepdim=True).values
e = torch.exp(x - m)
sigma = e.sum(dim=-1, keepdim=True)
y = e / sigma
loss = (y * w).sum()
loss.backward()
x.grad

tensor([[-0.1616,  0.0065, -0.0020,  0.1571],
        [-0.0425,  0.0790, -0.2612,  0.2247]])

In [9]:
torch.manual_seed(0)
x = torch.randn(2, 4)
w = torch.randn(2, 4)

m = x.max(dim=-1, keepdim=True).values
z = x - m
e = torch.exp(z)
sigma = e.sum(dim=-1, keepdim=True)
y = e / sigma
mul = y * w
loss = mul.sum()

dmul = torch.ones_like(mul)
print(f'{dmul.shape == mul.shape=}')
dy = w * dmul
print(f'{dy.shape == y.shape=}')
de = (1 / sigma) * dy
print(f'{de.shape == e.shape=}')
dsigma = ((-e/sigma**2) * dy).sum(dim=-1, keepdim=True)
print(f'{dsigma.shape == sigma.shape=}')
de += torch.ones_like(e) * dsigma
print(f'{de.shape == e.shape=}')
dz = e * de
print(f'{dz.shape == z.shape=}')
dx = 1 * dz
print(f'{dx.shape == x.shape=}')
dx

dmul.shape == mul.shape=True
dy.shape == y.shape=True
de.shape == e.shape=True
dsigma.shape == sigma.shape=True
de.shape == e.shape=True
dz.shape == z.shape=True
dx.shape == x.shape=True


tensor([[-0.1616,  0.0065, -0.0020,  0.1571],
        [-0.0425,  0.0790, -0.2612,  0.2247]])

In [10]:
torch.manual_seed(0)
x = torch.randn(2, 4)
w = torch.randn(2, 4)

m = x.max(dim=-1, keepdim=True).values
z = x - m
e = torch.exp(z)
sigma = e.sum(dim=-1, keepdim=True)
y = e / sigma
mul = y * w
loss = mul.sum()

dmul = torch.ones_like(mul)
print(f'{dmul.shape == mul.shape=}')
dy = w * dmul
print(f'{dy.shape == y.shape=}')
# remove intermediate steps to get one dx in terms of dy, e and sigma
# e/sigma get converted to y
# so final eq dx is in terms of y and dy
dx = e * ((1 / sigma) * dy + torch.ones_like(e) * ((-e/sigma**2) * dy).sum(dim=-1, keepdim=True))
print(dx)
dx = e * ((1 / sigma) * dy + ((-e/sigma**2) * dy).sum(dim=-1, keepdim=True))
print(dx)
dx = (e / sigma) * dy + e * ((-e/sigma**2) * dy).sum(dim=-1, keepdim=True)
print(dx)
dx = y * dy + e * ((-e/sigma**2) * dy).sum(dim=-1, keepdim=True)
print(dx)
dx = y * dy - e * ((e/sigma**2) * dy).sum(dim=-1, keepdim=True)
print(dx)
dx = y * dy - (e/sigma) * ((e/sigma) * dy).sum(dim=-1, keepdim=True)
print(dx)
dx = y * dy - y * (y * dy).sum(dim=-1, keepdim=True)
print(dx)
dx = y * (dy - (y * dy).sum(dim=-1, keepdim=True))
print(dx)
dx = y * (dy - (dy * y).sum(dim=-1, keepdim=True))
print(dx)
print(f'{dx.shape == x.shape=}')
dx

dmul.shape == mul.shape=True
dy.shape == y.shape=True
tensor([[-0.1616,  0.0065, -0.0020,  0.1571],
        [-0.0425,  0.0790, -0.2612,  0.2247]])
tensor([[-0.1616,  0.0065, -0.0020,  0.1571],
        [-0.0425,  0.0790, -0.2612,  0.2247]])
tensor([[-0.1616,  0.0065, -0.0020,  0.1571],
        [-0.0425,  0.0790, -0.2612,  0.2247]])
tensor([[-0.1616,  0.0065, -0.0020,  0.1571],
        [-0.0425,  0.0790, -0.2612,  0.2247]])
tensor([[-0.1616,  0.0065, -0.0020,  0.1571],
        [-0.0425,  0.0790, -0.2612,  0.2247]])
tensor([[-0.1616,  0.0065, -0.0020,  0.1571],
        [-0.0425,  0.0790, -0.2612,  0.2247]])
tensor([[-0.1616,  0.0065, -0.0020,  0.1571],
        [-0.0425,  0.0790, -0.2612,  0.2247]])
tensor([[-0.1616,  0.0065, -0.0020,  0.1571],
        [-0.0425,  0.0790, -0.2612,  0.2247]])
tensor([[-0.1616,  0.0065, -0.0020,  0.1571],
        [-0.0425,  0.0790, -0.2612,  0.2247]])
dx.shape == x.shape=True


tensor([[-0.1616,  0.0065, -0.0020,  0.1571],
        [-0.0425,  0.0790, -0.2612,  0.2247]])

### Arbitrary Gradient

can you guess why loss.backward() and passing g didn't make a difference in x.grad here?

answer in last cell

In [11]:
torch.manual_seed(0)
x = torch.randn(2, 4, requires_grad=True)

y = torch.softmax(x, dim=-1)
g = torch.randn(2, 4)
y.backward(g)
x.grad

tensor([[-0.1616,  0.0065, -0.0020,  0.1571],
        [-0.0425,  0.0790, -0.2612,  0.2247]])

In [12]:
dx = y * (g - (g * y).sum(dim=-1, keepdim=True))
dx

tensor([[-0.1616,  0.0065, -0.0020,  0.1571],
        [-0.0425,  0.0790, -0.2612,  0.2247]], grad_fn=<MulBackward0>)

In [13]:
# dy = w * dmul
# dmul is just ones
# dy = w
# since, w == g
# dy = g 